# Pilot simulation, Poisson version: target-level stability under photon-counting noise

**Purpose.** The first pilot assumed Gaussian noise. Real astronomical detectors count photons, so the noise is **Poisson**: its variance equals the expected count, and faint regions are noisier relative to their signal. This notebook repeats the pilot under Poisson noise and asks:

1. Does the main result still hold? *Star flux should pay a much smaller penalty for unknown gain than the least stable image direction.*
2. Does the stability measure κ_φ, now computed from the **Poisson Fisher information**, still predict the real scatter?
3. How much is lost by ignoring photon statistics, i.e. by fitting with unweighted least squares as if the noise were Gaussian?
4. What happens at **low light levels**, where counts are small and the linearized theory may break down?

**Model**

$$y_k \sim \mathrm{Poisson}(\mu_k), \qquad \mu_k = \mathrm{diag}(g)\,A_k x, \qquad g = 1 + B\theta$$

Everything else (sky basis, dithers, gain modes, targets) is identical to the Gaussian pilot.

**What changes in the theory.** For Poisson data the Fisher information is $F = J^\top \mathrm{diag}(1/\mu)\, J$, where $J$ is the Jacobian of the mean $\mu$. So the **whitened Jacobian** is $\mathrm{diag}(\mu^{-1/2})\,J$, and every formula from the Gaussian pilot (estimability, κ_φ, the Schur-complement split) applies unchanged to this whitened Jacobian. This is exactly the extension promised in Section A of the statement.

**Sections**
1. Setup and simulated data
2. Part 1: linearized theory with Poisson Fisher information
3. Part 2: Poisson maximum likelihood vs. ignoring photon statistics vs. assuming a flat gain
4. Part 3: low-light stress test
5. Part 4: conformal intervals under Poisson noise (optional)
6. Part 5: robustness across random seeds
7. Writing the result, limitations, interview questions

**Requirements:** `numpy`, `pandas`, `matplotlib`. Run all cells top to bottom (Kernel → Restart & Run All). Runtime is about 2 to 4 minutes.

## 1. Setup

Values are in **photon counts**. `SKY = 100` means 100 expected counts per pixel from the sky background, so its Poisson noise is about 10 counts. `LEVEL` scales every source of light at once (sky, extended emission, stars); Part 3 varies it to mimic shorter or longer exposures.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, LogLocator

# ---- configuration ----------------------------------------------------------
N_DET = 32            # detector is N_DET x N_DET pixels
MAX_SHIFT = 4         # largest dither offset in pixels
S = N_DET + MAX_SHIFT # scene grid size
N_STARS = 8
PSF_SIGMA = 1.2       # Gaussian PSF width in pixels
SKY = 100.0           # sky background, expected counts per pixel (at LEVEL = 1)
EXT_AMP = 10.0        # amplitude scale of smooth extended emission (keeps sky positive)
FLUX_RANGE = (500.0, 5000.0)   # total counts per star (at LEVEL = 1)
GAIN_RMS = 0.03       # rms of each gain-mode coefficient
EXT_ORDER = 3
GAIN_ORDER = 2
DITHER_AMPS = [0, 1, 2, 3, 4]

SEED = 0
MC_TRIALS = 300       # Monte Carlo repetitions in Part 2
MC_AMP = 2            # dither amplitude used in Parts 2 to 5
LEVELS = [0.1, 0.3, 1.0, 3.0]   # light levels for the low-light stress test (Part 3)
STRESS_TRIALS = 200
RUN_CONFORMAL = True
N_CASES = 800

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"

### Building blocks
Same as the Gaussian pilot, with three changes:
- `Problem(..., level)` scales all light by `level`.
- `simulate` draws **Poisson** counts.
- `whitened_jacobian` divides each row by $\sqrt{\mu}$, the Poisson standard deviation at that pixel.

In [ ]:
def cosine_modes(size, max_order):
    idx = (np.arange(size) + 0.5) / size
    modes = []
    for u in range(max_order + 1):
        for v in range(max_order + 1):
            if 1 <= u + v <= max_order:
                m = np.outer(np.cos(np.pi * u * idx), np.cos(np.pi * v * idx))
                modes.append((m / np.sqrt(np.mean(m ** 2))).ravel())
    return np.array(modes).T

def star_templates(positions):
    yy, xx = np.mgrid[0:S, 0:S]
    cols = []
    for (py, px) in positions:
        t = np.exp(-((yy - py) ** 2 + (xx - px) ** 2) / (2 * PSF_SIGMA ** 2))
        cols.append((t / t.sum()).ravel())
    return np.array(cols).T

def random_positions(rng):
    lo, hi = MAX_SHIFT + 3, N_DET - 3
    pos = []
    while len(pos) < N_STARS:
        p = rng.uniform(lo, hi, size=2)
        if all(np.hypot(*(p - q)) > 4 * PSF_SIGMA for q in pos):
            pos.append(p)
    return np.array(pos)

def dither_offsets(amp):
    return [(0, 0), (amp, 0), (0, amp), (amp, amp)]

def window(img_flat, off):
    dy, dx = off
    return img_flat.reshape(S, S)[dy:dy + N_DET, dx:dx + N_DET].ravel()

class Problem:
    def __init__(self, rng, amp, level=1.0):
        self.pos = random_positions(rng)
        ext = cosine_modes(S, EXT_ORDER)
        self.V = np.hstack([np.ones((S * S, 1)), ext, star_templates(self.pos)])
        self.n_ext = ext.shape[1]
        self.ext_idx = np.arange(1, 1 + self.n_ext)
        self.star_idx = np.arange(1 + self.n_ext, self.V.shape[1])
        self.B = cosine_modes(N_DET, GAIN_ORDER)
        self.offs = dither_offsets(amp)
        self.AV = [np.array([window(self.V[:, j], o) for j in range(self.V.shape[1])]).T
                   for o in self.offs]
        while True:   # redraw the extended emission if the sky would dip below 20% of its mean level
            ext_coef = EXT_AMP * rng.standard_normal(self.n_ext)
            if (SKY + ext @ ext_coef).min() > 0.2 * SKY:
                break
        self.a_true = level * np.concatenate([[SKY], ext_coef, rng.uniform(*FLUX_RANGE, size=N_STARS)])
        self.th_true = GAIN_RMS * rng.standard_normal(self.B.shape[1])

    def gain(self, th):
        return 1.0 + self.B @ th

    def mean(self, a, th):
        g = self.gain(th)
        return np.concatenate([g * (AV @ a) for AV in self.AV])

    def jacobian(self, a, th):
        # Jacobian of the mean (not whitened)
        g = self.gain(th)
        Ja = np.vstack([g[:, None] * AV for AV in self.AV])
        Jt = np.vstack([(AV @ a)[:, None] * self.B for AV in self.AV])
        return Ja, Jt

    def whitened_jacobian(self, a, th):
        mu = np.maximum(self.mean(a, th), 1e-9)
        Ja, Jt = self.jacobian(a, th)
        w = 1.0 / np.sqrt(mu)
        return w[:, None] * Ja, w[:, None] * Jt

    def simulate(self, rng):
        return rng.poisson(self.mean(self.a_true, self.th_true)).astype(float)

### Look at the simulated data
The right panel is one exposure in **photon counts**. Compare the graininess with the Gaussian pilot: Poisson noise is larger where the image is brighter.

In [ ]:
rng = np.random.default_rng(SEED)
demo = Problem(rng, MC_AMP)
sky = (demo.V @ demo.a_true).reshape(S, S)
g_true = demo.gain(demo.th_true).reshape(N_DET, N_DET)
y_demo = demo.simulate(rng)[:N_DET * N_DET].reshape(N_DET, N_DET)

fig, axs = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, (img, title) in zip(axs, [(sky, "True sky (expected counts)"), (g_true, "True detector gain"),
                                  (y_demo, "One Poisson exposure (counts)")]):
    im = ax.imshow(img, origin="lower", cmap="viridis")
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
axs[0].scatter(demo.pos[:, 1], demo.pos[:, 0], s=90, facecolors="none", edgecolors="white", lw=1)
plt.tight_layout(); plt.show()
print(f"Minimum expected count: {demo.mean(demo.a_true, demo.th_true).min():.1f} (must be positive)")
print(f"Star fluxes (total counts): {np.round(demo.a_true[demo.star_idx]).astype(int).tolist()}")

## 2. Part 1: Linearized theory with Poisson Fisher information

Identical to the Gaussian pilot, except the Jacobian is whitened by $\mu^{-1/2}$. Columns:
- `all stars estimable`: the target lies in the row space of the whitened Jacobian.
- `star inflation`: variance with unknown gain ÷ variance with known gain. **1.0 means no penalty.**
- `worst direction inflation`: the largest inflation over every linear combination of image coefficients.
- `max formula rel. error`: direct inversion vs. the Schur-complement formula in the statement. Should be tiny.

In [ ]:
def kappa_analysis(prob, a, th, compute_worst=True):
    Ja, Jt = prob.whitened_jacobian(a, th)
    J = np.hstack([Ja, Jt])
    p = Ja.shape[1]
    Fa = Ja.T @ Ja
    Sig_known = np.linalg.inv(Fa)
    Sig_joint = np.linalg.pinv(J.T @ J)[:p, :p]
    M = np.linalg.solve(Fa, Ja.T @ Jt)
    C = Jt - Ja @ M
    _, s, Vt = np.linalg.svd(C, full_matrices=False)
    rank = np.linalg.matrix_rank(J)
    P_row = np.linalg.pinv(J) @ J
    rows, I = [], np.eye(p)
    labels = [(i, f"star {k}") for k, i in enumerate(prob.star_idx)] + \
             [(i, f"ext mode {k}") for k, i in enumerate(prob.ext_idx)]
    for idx, label in labels:
        e = I[idx]
        avec = np.concatenate([e, np.zeros(Jt.shape[1])])
        estimable = np.linalg.norm(avec - P_row @ avec) < 1e-8 * np.linalg.norm(avec)
        k2_known, k2_joint = e @ Sig_known @ e, e @ Sig_joint @ e
        calib = np.sum(((e @ M) @ Vt.T) ** 2 / s ** 2) if rank == J.shape[1] else np.nan
        rows.append({"target": label, "estimable": estimable,
                     "kappa_known": np.sqrt(k2_known), "kappa_joint": np.sqrt(k2_joint),
                     "formula_rel_err": abs(np.sqrt(k2_known + calib) - np.sqrt(k2_joint)) / np.sqrt(k2_joint),
                     "inflation": k2_joint / k2_known})
    worst = np.nan
    if compute_worst:
        w, U = np.linalg.eigh(Sig_known)
        Sk_mhalf = U @ np.diag(w ** -0.5) @ U.T
        worst = np.linalg.eigvalsh(Sk_mhalf @ Sig_joint @ Sk_mhalf).max()
    return pd.DataFrame(rows), {"rank": rank, "n_params": J.shape[1], "s_min_C": s.min(),
                                "worst_inflation": worst}

summary = []
for amp in DITHER_AMPS:
    prob = Problem(np.random.default_rng(SEED), amp)
    df, info = kappa_analysis(prob, prob.a_true, prob.th_true)
    st, ex = df[df.target.str.startswith("star")], df[df.target.str.startswith("ext")]
    summary.append({"dither amp (px)": amp, "rank / params": f"{info['rank']}/{info['n_params']}",
                    "all stars estimable": bool(st.estimable.all()),
                    "star inflation (median)": st.inflation.median(),
                    "star inflation (max)": st.inflation.max(),
                    "ext inflation (median)": ex.inflation.median(),
                    "worst direction inflation": info["worst_inflation"],
                    "max formula rel. error": df.formula_rel_err.max()})
part1 = pd.DataFrame(summary)
display(part1.round({"star inflation (median)": 2, "star inflation (max)": 2,
                     "ext inflation (median)": 2, "worst direction inflation": 1}))

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.8), dpi=120)
ax.plot(part1["dither amp (px)"], part1["worst direction inflation"], color=ORANGE, lw=2,
        marker="o", ms=6, label="Worst image direction")
ax.plot(part1["dither amp (px)"], part1["star inflation (median)"], color=BLUE, lw=2,
        marker="o", ms=6, label="Star flux (median)")
ax.set_yscale("log")
ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5)))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.yaxis.set_minor_formatter(FuncFormatter(lambda v, _: ""))
ax.set_xticks(DITHER_AMPS)
ax.set_xlabel("Dither amplitude (pixels)")
ax.set_ylabel("Variance inflation from unknown gain")
ax.set_title("Poisson noise: star flux vs. worst image direction", fontsize=10)
ax.grid(True, color="#e6e6e6", lw=0.8)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); fig.savefig("poisson_inflation_vs_dither.png", dpi=200); plt.show()

## 3. Part 2: Three estimators on Poisson data

- **Poisson ML (joint):** maximizes the Poisson likelihood over image and gain by Fisher scoring. This is the method the theory describes.
- **Unweighted least squares (joint):** also estimates image and gain, but **ignores photon statistics**, treating every pixel as equally noisy. A common shortcut.
- **Flat-gain Poisson ML:** respects photon statistics but **assumes the gain is flat** ($g = 1$).

| Column | Meaning |
|---|---|
| `ML rel. RMSE %` | Typical flux error of Poisson ML, as a percent of true flux |
| `ML bias %` | Average error of Poisson ML; should be near zero |
| `pred/emp sd` | κ from Poisson Fisher information ÷ observed scatter; **near 1.0 means the theory predicts reality** |
| `LS / ML variance` | How much larger the variance is when photon statistics are ignored; **above 1 means ignoring them costs precision** |
| `flat-gain bias %` | Average error when the gain is wrongly assumed flat |

In [ ]:
def loglik(prob, y, a, th):
    mu = prob.mean(a, th)
    return -np.inf if np.any(mu <= 0) else np.sum(y * np.log(mu) - mu)

def fisher_scoring(prob, y, a, th, fit_gain=True, iters=60):
    ll = loglik(prob, y, a, th)
    for _ in range(iters):
        mu = np.maximum(prob.mean(a, th), 1e-9)
        Ja, Jt = prob.jacobian(a, th)
        J = np.hstack([Ja, Jt]) if fit_gain else Ja
        w = 1.0 / np.sqrt(mu)
        step = np.linalg.lstsq(w[:, None] * J, (y - mu) * w, rcond=None)[0]
        t = 1.0
        while t > 1e-6:   # backtracking: accept only steps that raise the likelihood
            a_new = a + t * step[:a.size]
            th_new = th + t * step[a.size:] if fit_gain else th
            ll_new = loglik(prob, y, a_new, th_new)
            if ll_new >= ll:
                break
            t /= 2
        if t <= 1e-6:
            break
        a, th, ll = a_new, th_new, ll_new
        if np.linalg.norm(t * step) < 1e-10 * (1 + np.linalg.norm(a)):
            break
    return a, th

def init_ls(prob, y):
    return np.linalg.lstsq(np.vstack(prob.AV), y, rcond=None)[0]

def estimate_poisson_ml(prob, y):
    return fisher_scoring(prob, y, init_ls(prob, y), np.zeros(prob.B.shape[1]))

def estimate_flat_gain(prob, y):
    return fisher_scoring(prob, y, init_ls(prob, y), np.zeros(prob.B.shape[1]), fit_gain=False)

def estimate_unweighted_ls(prob, y, iters=40):
    a, th = init_ls(prob, y), np.zeros(prob.B.shape[1])
    for _ in range(iters):
        r = y - prob.mean(a, th)
        Ja, Jt = prob.jacobian(a, th)
        step = np.linalg.lstsq(np.hstack([Ja, Jt]), r, rcond=None)[0]
        a, th = a + step[:a.size], th + step[a.size:]
        if np.linalg.norm(step) < 1e-10 * (1 + np.linalg.norm(a)):
            break
    return a, th

def monte_carlo(seed, amp, trials, level=1.0, with_ls=True):
    rng = np.random.default_rng(seed)
    prob = Problem(rng, amp, level)
    df, info = kappa_analysis(prob, prob.a_true, prob.th_true)
    kap = df[df.target.str.startswith("star")].kappa_joint.values
    e_ml, e_ls, e_flat = [], [], []
    for _ in range(trials):
        y = prob.simulate(rng)
        e_ml.append(estimate_poisson_ml(prob, y)[0][prob.star_idx] - prob.a_true[prob.star_idx])
        e_flat.append(estimate_flat_gain(prob, y)[0][prob.star_idx] - prob.a_true[prob.star_idx])
        if with_ls:
            e_ls.append(estimate_unweighted_ls(prob, y)[0][prob.star_idx] - prob.a_true[prob.star_idx])
    e_ml, e_flat = np.array(e_ml), np.array(e_flat)
    flux = prob.a_true[prob.star_idx]
    out = pd.DataFrame({"flux (counts)": flux,
                        "ML rel. RMSE %": 100 * np.sqrt((e_ml ** 2).mean(0)) / flux,
                        "ML bias %": 100 * e_ml.mean(0) / flux,
                        "pred/emp sd": kap / e_ml.std(0),
                        "flat-gain bias %": 100 * e_flat.mean(0) / flux})
    if with_ls:
        out.insert(4, "LS / ML variance", np.array(e_ls).var(0) / e_ml.var(0))
    st = df[df.target.str.startswith("star")]
    return out, {"star_inflation_median": st.inflation.median(), "worst_inflation": info["worst_inflation"]}

part2, _ = monte_carlo(SEED, MC_AMP, MC_TRIALS)
print(f"Dither amplitude {MC_AMP}, {MC_TRIALS} trials, LEVEL = 1")
display(part2.round({"flux (counts)": 0, "ML rel. RMSE %": 3, "ML bias %": 3, "pred/emp sd": 3,
                     "LS / ML variance": 2, "flat-gain bias %": 2}))

**Sanity checks:** `pred/emp sd` should be within about ±0.1 of 1, `ML bias %` near zero, and `LS / ML variance` at or above about 1. Poisson ML is the efficient estimator, so ignoring photon statistics should never help on average.

## 4. Part 3: Low-light stress test

`LEVEL` scales all light. At `LEVEL = 0.1`, the sky is about 10 counts per pixel and the faintest star has about 50 counts in total: a short exposure. Here the **linearized theory may start to fail**, because the Poisson likelihood is far from Gaussian at low counts. This is a stress test; we want to see *where* it breaks, not to hide it.

In [ ]:
stress = []
for level in LEVELS:
    mc, th = monte_carlo(SEED, MC_AMP, STRESS_TRIALS, level=level, with_ls=False)
    stress.append({"LEVEL": level, "sky counts/pixel": SKY * level,
                   "faintest star (counts)": mc["flux (counts)"].min(),
                   "star inflation (median)": th["star_inflation_median"],
                   "worst direction inflation": th["worst_inflation"],
                   "ML |bias| % (max)": mc["ML bias %"].abs().max(),
                   "ML rel. RMSE % (max)": mc["ML rel. RMSE %"].max(),
                   "pred/emp sd (min)": mc["pred/emp sd"].min(),
                   "pred/emp sd (max)": mc["pred/emp sd"].max(),
                   "flat-gain |bias| % (max)": mc["flat-gain bias %"].abs().max()})
stress = pd.DataFrame(stress)
display(stress.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.6), dpi=120)
ax.plot(stress["sky counts/pixel"], stress["pred/emp sd (min)"], color=BLUE, lw=2, marker="o", ms=6,
        label="pred/emp sd (min over stars)")
ax.plot(stress["sky counts/pixel"], stress["pred/emp sd (max)"], color=AQUA, lw=2, marker="o", ms=6,
        label="pred/emp sd (max over stars)")
ax.axhline(1.0, color="#999999", lw=1, ls="--")
ax.set_ylim(0.7, 1.3)
ax.set_xscale("log")
ax.set_xticks(stress["sky counts/pixel"]); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xlabel("Sky background (counts per pixel)")
ax.set_ylabel("Predicted / observed scatter")
ax.set_title("Does the Poisson theory hold at low light?", fontsize=10)
ax.grid(True, color="#e6e6e6", lw=0.8)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); fig.savefig("poisson_low_light.png", dpi=200); plt.show()

**How to read the stress test**
- `star inflation` and `worst direction inflation` do not change with `LEVEL`. That is expected: scaling all light scales the Fisher information uniformly, so the *relative* cost of unknown gain is the same. What changes with light level is the absolute error and whether the linearized theory still holds.
- If `pred/emp sd` stays near 1 down to the lowest level, the theory is robust to low counts.
- If it drifts away from 1 at low levels, **that is a finding, not a failure**: it marks where the local theory needs the finite-sample bounds promised in Section B. Report it honestly.
- `ML |bias| %` growing at low counts is expected; maximum likelihood is only asymptotically unbiased.

## 5. Part 4 (optional): Conformal intervals under Poisson noise

Same design as the Gaussian pilot: independent cases, scores |error| / κ with κ computed **at the estimate** from the Poisson Fisher information, half for calibration, half for testing. Conformal coverage does not depend on the noise model, so coverage should again be close to 0.95. What may change is the interval width.

In [ ]:
def conformal(seed, amp, n_cases, alpha=0.05):
    rng = np.random.default_rng(seed + 1)
    rows = []
    for _ in range(n_cases):
        prob = Problem(rng, amp)
        y = prob.simulate(rng)
        a_hat, th_hat = estimate_poisson_ml(prob, y)
        df, _ = kappa_analysis(prob, a_hat, th_hat, compute_worst=False)
        kap = df[df.target.str.startswith("star")].kappa_joint.values
        rows.append((np.abs(a_hat[prob.star_idx] - prob.a_true[prob.star_idx]) / kap,
                     prob.a_true[prob.star_idx]))
    half = n_cases // 2
    cal, test = rows[:half], rows[half:]
    def qhat(scores):
        n = len(scores); k = int(np.ceil((n + 1) * (1 - alpha)))
        return np.sort(scores)[min(k, n) - 1]
    q_marg = qhat(np.array([r[0][rng.integers(N_STARS)] for r in cal]))
    q_sim = qhat(np.array([r[0].max() for r in cal]))
    ts = np.concatenate([r[0] for r in test]); tf = np.concatenate([r[1] for r in test])
    cover = ts <= q_marg
    cover_sim = np.array([r[0].max() <= q_sim for r in test])
    edges = np.quantile(tf, [0, 1/3, 2/3, 1])
    by_bin = pd.DataFrame([{"flux bin (counts)": f"{lo:.0f} to {hi:.0f}",
                            "per-star coverage": cover[(tf >= lo) & (tf <= hi)].mean()}
                           for lo, hi in zip(edges[:-1], edges[1:])])
    overall = pd.DataFrame([{"interval type": "per star", "half-width (x kappa)": q_marg, "coverage": cover.mean()},
                            {"interval type": "simultaneous (whole field)", "half-width (x kappa)": q_sim,
                             "coverage": cover_sim.mean()}])
    return overall, by_bin

if RUN_CONFORMAL:
    overall, by_bin = conformal(SEED, MC_AMP, N_CASES)
    display(overall.round(3)); display(by_bin.round(3))

**Reading this:** a `half-width` near 2 means the intervals are about ±2κ, which is what you would expect if κ is an accurate standard error. A much larger half-width would mean κ underestimates the real error.

## 6. Part 5: Robustness across random seeds

In [ ]:
rob = []
for seed in range(5):
    mc, th = monte_carlo(seed, MC_AMP, 100)
    rob.append({"seed": seed,
                "star inflation (median)": th["star_inflation_median"],
                "worst direction inflation": th["worst_inflation"],
                "ratio worst / star": th["worst_inflation"] / th["star_inflation_median"],
                "ML rel. RMSE % (max)": mc["ML rel. RMSE %"].max(),
                "LS / ML variance (median)": mc["LS / ML variance"].median(),
                "flat-gain |bias| % (max)": mc["flat-gain bias %"].abs().max(),
                "pred/emp sd (min)": mc["pred/emp sd"].min(),
                "pred/emp sd (max)": mc["pred/emp sd"].max()})
robust = pd.DataFrame(rob)
display(robust.round(2))
print("Median across seeds:")
print(robust.drop(columns="seed").median().round(2).to_string())

## 7. Writing the result

Use **your own numbers**, and describe what you actually observe. The Poisson result most naturally extends the existing pilot sentence in Section B. Possible additions:

> ...and the same pattern held under Poisson photon-counting noise (median star-flux inflation **[A]** versus **[B]**), where κ_φ from the Poisson Fisher information predicted the observed scatter within **[C]%**.

> ...under Poisson noise, ignoring photon statistics increased star-flux variance by a median factor of **[D]**.

If Part 3 shows the theory drifting at low counts, a strong and honest addition is:

> ...the linearized prediction held down to about **[E]** sky counts per pixel and degraded below it, motivating the finite-sample bounds in this aim.

**Be honest.** If a check fails or a number is unflattering, report it as it is. A clearly stated limit is more convincing to an expert reviewer than a claim that everything works.

### Sanity checklist
1. `max formula rel. error` is tiny (Part 1).
2. At `LEVEL = 1`, `pred/emp sd` is close to 1 and `ML bias %` is near zero (Part 2).
3. `LS / ML variance` is at or above about 1 (Part 2).
4. The pattern holds across seeds (Part 5).

### Limitations (be ready to state these)
- Pure Poisson noise; real detectors also add read noise, which is Gaussian. A Poisson-plus-Gaussian model is a natural next step.
- Star positions and the point-spread function are known.
- Integer-pixel dithers, small detector, smooth gain only (no pixel-level component η).
- The theory is local and asymptotic; Part 3 shows where it starts to break down.

### Questions you may be asked in an interview
- Why does the Poisson Fisher information use the weights $1/\mu$? *(Hint: Poisson variance equals the mean.)*
- Why can ignoring photon statistics lose precision for stars? *(Hint: unweighted least squares gives bright, noisy star peaks the same weight as quiet sky pixels.)*
- Why does maximum likelihood become biased at very low counts?
- How would you add read noise to this model?
- Why does conformal coverage still hold even when the noise model changes?